In [0]:
%run ./config

In [0]:
%pip install azure-storage-file-datalake faker


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 42.4 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
"""
Retail Analytics Platform — Test Data Generator
================================================
Generates realistic but intentionally dirty data for:
  - customers  → ADLS landing/customers/
  - products   → ADLS landing/products/
  - orders     → ADLS landing/orders/

Bad data breakdown (~300-700 rejects per table):
  customers : NULL CustomerID, invalid Email, empty FirstName/LastName
  products  : NULL ProductID, empty ProductName, negative/zero CostPrice
  orders    : NULL OrderID, negative Quantity/UnitPrice, future OrderDate,
              FK violations (invalid CustomerID / ProductID)

Run:
    pip install azure-storage-file-datalake faker
    python generate_data.py

Change RUN_NUMBER = 1 for Day-2, RUN_NUMBER = 3 for Day-3, and so on.
"""

import csv
import io
import random
from datetime import date, timedelta
from faker import Faker
from azure.storage.filedatalake import DataLakeServiceClient

# ── CONFIG — only change these ────────────────────────────────────────────────
RUN_NUMBER           = 2               # Day 1 → 1, Day 2 → 2, Day 3 → 3 ...

STORAGE_ACCOUNT_NAME = storage_account_name
STORAGE_ACCOUNT_KEY  = storage_account_key
CONTAINER_NAME       = container_name
# ─────────────────────────────────────────────────────────────────────────────

TOTAL_ROWS = 2000
BAD_ROWS   = random.randint(300, 700)
GOOD_ROWS  = TOTAL_ROWS - BAD_ROWS

fake       = Faker()
random.seed(RUN_NUMBER * 42)
TODAY      = date.today()

# ID ranges — new IDs per day + 200 overlap for SCD2 updates
OVERLAP           = 200
STEP              = GOOD_ROWS - OVERLAP
CUSTOMER_ID_START = 1 + (RUN_NUMBER - 1) * STEP
PRODUCT_ID_START  = 1 + (RUN_NUMBER - 1) * STEP
ORDER_ID_START    = 1 + (RUN_NUMBER - 1) * TOTAL_ROWS  # orders never overlap

# ADLS landing paths
CUSTOMERS_ADLS_PATH = f"landing/customers/customers_run{RUN_NUMBER}.csv"
PRODUCTS_ADLS_PATH  = f"landing/products/products_run{RUN_NUMBER}.csv"
ORDERS_ADLS_PATH    = f"landing/orders/orders_run{RUN_NUMBER}.csv"

STORE_CODES = [f"ST{str(i).zfill(3)}" for i in range(1, 51)]

CURRENCY_CODES = [
    "USD","EUR","GBP","INR","JPY","AUD","CAD","CHF","CNY","SGD",
    "AED","SAR","QAR","KWD","OMR","NZD","HKD","THB","MYR","IDR",
    "PHP","KRW","VND","PKR","BDT","LKR","NPR","ZAR","NGN","EGP",
    "TRY","RUB","MXN","BRL","ARS","CLP","COP","PEN","UYU","CZK",
    "DKK","NOK","SEK","PLN","HUF","RON","BGN","UAH","MAD","KES"
]

# ─────────────────────────────────────────────────────────────────────────────
# ADLS UPLOAD
# ─────────────────────────────────────────────────────────────────────────────

def upload_to_adls(content_str, adls_path):
    client      = DataLakeServiceClient(
                    account_url=f"https://{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net",
                    credential=STORAGE_ACCOUNT_KEY
                  )
    fs_client   = client.get_file_system_client(CONTAINER_NAME)
    file_client = fs_client.get_file_client(adls_path)
    file_client.upload_data(content_str.encode("utf-8"), overwrite=True)
    print(f"  ✓ Uploaded → adls://{CONTAINER_NAME}/{adls_path}")

# ─────────────────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def rand_date_str(start_days_ago=730, end_days_ago=1):
    delta = random.randint(end_days_ago, start_days_ago)
    return (TODAY - timedelta(days=delta)).strftime("%Y-%m-%d")

def rand_timestamp():
    delta = random.randint(1, 730)
    dt    = TODAY - timedelta(days=delta)
    return (f"{dt.strftime('%Y-%m-%d')} "
            f"{random.randint(0,23):02d}:"
            f"{random.randint(0,59):02d}:"
            f"{random.randint(0,59):02d}")

# ─────────────────────────────────────────────────────────────────────────────
# 1. CUSTOMERS → ADLS landing/customers/
# ─────────────────────────────────────────────────────────────────────────────

def generate_customers():
    rows               = []
    valid_customer_ids = []

    # Good rows
    for i in range(GOOD_ROWS):
        cid = CUSTOMER_ID_START + i
        valid_customer_ids.append(cid)
        rows.append({
            "CustomerID" : cid,
            "FirstName"  : fake.first_name(),
            "LastName"   : fake.last_name(),
            "Email"      : fake.email(),
            "Phone"      : fake.phone_number()[:20],
            "City"       : fake.city(),
            "State"      : fake.state(),
            "LastUpdated": rand_timestamp()
        })

    # Bad rows
    bad_types = [
        "null_pk",          # empty CustomerID   → PK fail
        "null_email",       # empty Email        → not_null fail
        "invalid_email",    # bad email format   → valid_email fail
        "empty_firstname",  # empty FirstName    → not_empty fail
        "empty_lastname",   # empty LastName     → not_empty fail
        "duplicate_pk",     # duplicate ID       → downstream issue
    ]

    for i in range(BAD_ROWS):
        bad_type = bad_types[i % len(bad_types)]
        cid      = CUSTOMER_ID_START + GOOD_ROWS + i
        row = {
            "CustomerID" : cid,
            "FirstName"  : fake.first_name(),
            "LastName"   : fake.last_name(),
            "Email"      : fake.email(),
            "Phone"      : fake.phone_number()[:20],
            "City"       : fake.city(),
            "State"      : fake.state(),
            "LastUpdated": rand_timestamp()
        }
        if bad_type == "null_pk":
            row["CustomerID"] = ""
        elif bad_type == "null_email":
            row["Email"] = ""
        elif bad_type == "invalid_email":
            row["Email"] = random.choice([
                "notanemail", "missing@", "@nodomain.com",
                "spaces in@email.com", "nodot@domain"
            ])
        elif bad_type == "empty_firstname":
            row["FirstName"] = ""
        elif bad_type == "empty_lastname":
            row["LastName"] = ""
        elif bad_type == "duplicate_pk":
            row["CustomerID"] = random.choice(valid_customer_ids[:100])
        rows.append(row)

    random.shuffle(rows)

    output = io.StringIO()
    writer = csv.DictWriter(output, fieldnames=[
        "CustomerID", "FirstName", "LastName",
        "Email", "Phone", "City", "State", "LastUpdated"
    ])
    writer.writeheader()
    writer.writerows(rows)
    upload_to_adls(output.getvalue(), CUSTOMERS_ADLS_PATH)

    return [r["CustomerID"] for r in rows if r["CustomerID"] != ""]

# ─────────────────────────────────────────────────────────────────────────────
# 2. PRODUCTS → ADLS landing/products/
# ─────────────────────────────────────────────────────────────────────────────

CATEGORIES = {
    "Electronics"   : ["Smartphones", "Laptops", "Tablets", "Headphones", "Cameras"],
    "Clothing"      : ["Men's Wear", "Women's Wear", "Kids", "Sportswear", "Accessories"],
    "Home & Kitchen": ["Cookware", "Furniture", "Bedding", "Appliances", "Decor"],
    "Books"         : ["Fiction", "Non-Fiction", "Academic", "Comics", "Children"],
    "Sports"        : ["Gym", "Outdoor", "Team Sports", "Water Sports", "Cycling"],
}
BRANDS = ["Samsung", "Apple", "Nike", "Sony", "LG", "Adidas", "Philips",
          "Panasonic", "Lenovo", "Dell", "Reebok", "Puma", "HP", "Bosch", "Ikea"]

def generate_products():
    rows              = []
    valid_product_ids = []

    # Good rows
    for i in range(GOOD_ROWS):
        pid      = PRODUCT_ID_START + i
        category = random.choice(list(CATEGORIES.keys()))
        sub      = random.choice(CATEGORIES[category])
        valid_product_ids.append(pid)
        rows.append({
            "ProductID"  : pid,
            "ProductName": f"{random.choice(BRANDS)} {fake.word().capitalize()} {sub} {pid}",
            "Category"   : category,
            "SubCategory": sub,
            "Brand"      : random.choice(BRANDS),
            "CostPrice"  : round(random.uniform(5.0, 2000.0), 2)
        })

    # Bad rows
    bad_types = [
        "null_pk",        # empty ProductID    → PK fail
        "empty_name",     # empty ProductName  → not_empty fail
        "zero_cost",      # CostPrice = 0      → positive fail
        "negative_cost",  # CostPrice < 0      → positive fail
        "null_cost",      # empty CostPrice    → positive fail
    ]

    for i in range(BAD_ROWS):
        bad_type = bad_types[i % len(bad_types)]
        pid      = PRODUCT_ID_START + GOOD_ROWS + i
        category = random.choice(list(CATEGORIES.keys()))
        sub      = random.choice(CATEGORIES[category])
        row = {
            "ProductID"  : pid,
            "ProductName": f"{random.choice(BRANDS)} {fake.word().capitalize()} {sub} {pid}",
            "Category"   : category,
            "SubCategory": sub,
            "Brand"      : random.choice(BRANDS),
            "CostPrice"  : round(random.uniform(5.0, 2000.0), 2)
        }
        if bad_type == "null_pk":
            row["ProductID"] = ""
        elif bad_type == "empty_name":
            row["ProductName"] = ""
        elif bad_type == "zero_cost":
            row["CostPrice"] = 0
        elif bad_type == "negative_cost":
            row["CostPrice"] = round(random.uniform(-500.0, -0.01), 2)
        elif bad_type == "null_cost":
            row["CostPrice"] = ""
        rows.append(row)

    random.shuffle(rows)

    output = io.StringIO()
    writer = csv.DictWriter(output, fieldnames=[
        "ProductID", "ProductName", "Category", "SubCategory", "Brand", "CostPrice"
    ])
    writer.writeheader()
    writer.writerows(rows)
    upload_to_adls(output.getvalue(), PRODUCTS_ADLS_PATH)

    return valid_product_ids

# ─────────────────────────────────────────────────────────────────────────────
# 3. ORDERS → ADLS landing/orders/
# ─────────────────────────────────────────────────────────────────────────────

def generate_orders(valid_customer_ids, valid_product_ids):
    fk_customer_pool = valid_customer_ids[:GOOD_ROWS]
    fk_product_pool  = valid_product_ids[:GOOD_ROWS]
    rows             = []

    # Good rows
    for i in range(GOOD_ROWS):
        rows.append({
            "OrderID"     : ORDER_ID_START + i,
            "CustomerID"  : random.choice(fk_customer_pool),
            "ProductID"   : random.choice(fk_product_pool),
            "OrderDate"   : rand_date_str(),
            "Quantity"    : random.randint(1, 50),
            "UnitPrice"   : round(random.uniform(1.0, 3000.0), 2),
            "StoreCode"   : random.choice(STORE_CODES),
            "CurrencyCode": random.choice(CURRENCY_CODES)
        })

    # Bad rows
    bad_types = [
        "null_pk",            # empty OrderID      → PK fail
        "null_orderdate",     # empty OrderDate    → not_null fail
        "future_orderdate",   # future OrderDate   → not_future fail
        "zero_quantity",      # Quantity = 0       → positive fail
        "negative_quantity",  # Quantity < 0       → positive fail
        "zero_unitprice",     # UnitPrice = 0      → positive fail
        "negative_unitprice", # UnitPrice < 0      → positive fail
        "bad_customer_fk",    # invalid CustomerID → FK fail
        "bad_product_fk",     # invalid ProductID  → FK fail
    ]

    for i in range(BAD_ROWS):
        bad_type = bad_types[i % len(bad_types)]
        row = {
            "OrderID"     : ORDER_ID_START + GOOD_ROWS + i,
            "CustomerID"  : random.choice(fk_customer_pool),
            "ProductID"   : random.choice(fk_product_pool),
            "OrderDate"   : rand_date_str(),
            "Quantity"    : random.randint(1, 50),
            "UnitPrice"   : round(random.uniform(1.0, 3000.0), 2),
            "StoreCode"   : random.choice(STORE_CODES),
            "CurrencyCode": random.choice(CURRENCY_CODES)
        }
        if bad_type == "null_pk":
            row["OrderID"] = ""
        elif bad_type == "null_orderdate":
            row["OrderDate"] = ""
        elif bad_type == "future_orderdate":
            row["OrderDate"] = (TODAY + timedelta(days=random.randint(1, 365))).strftime("%Y-%m-%d")
        elif bad_type == "zero_quantity":
            row["Quantity"] = 0
        elif bad_type == "negative_quantity":
            row["Quantity"] = random.randint(-50, -1)
        elif bad_type == "zero_unitprice":
            row["UnitPrice"] = 0
        elif bad_type == "negative_unitprice":
            row["UnitPrice"] = round(random.uniform(-500.0, -0.01), 2)
        elif bad_type == "bad_customer_fk":
            row["CustomerID"] = random.randint(99000, 99999)
        elif bad_type == "bad_product_fk":
            row["ProductID"] = random.randint(99000, 99999)
        rows.append(row)

    random.shuffle(rows)

    output = io.StringIO()
    writer = csv.DictWriter(output, fieldnames=[
        "OrderID", "CustomerID", "ProductID",
        "OrderDate", "Quantity", "UnitPrice", "StoreCode", "CurrencyCode"
    ])
    writer.writeheader()
    writer.writerows(rows)
    upload_to_adls(output.getvalue(), ORDERS_ADLS_PATH)

# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    print(f"\n{'='*55}")
    print(f"  Retail Data Generator — Run {RUN_NUMBER}")
    print(f"  Total rows per table : {TOTAL_ROWS}")
    print(f"  Expected rejects     : ~{BAD_ROWS} per table")
    print(f"{'='*55}\n")

    print("[1/3] Generating customers...")
    valid_customer_ids = generate_customers()

    print("[2/3] Generating products...")
    valid_product_ids = generate_products()

    print("[3/3] Generating orders...")
    generate_orders(valid_customer_ids, valid_product_ids)

    print(f"\n{'='*55}")
    print(f"  All done for Run {RUN_NUMBER}!")
    print(f"  customers → adls://{CONTAINER_NAME}/{CUSTOMERS_ADLS_PATH}")
    print(f"  products  → adls://{CONTAINER_NAME}/{PRODUCTS_ADLS_PATH}")
    print(f"  orders    → adls://{CONTAINER_NAME}/{ORDERS_ADLS_PATH}")
    print(f"\n  Next day → set RUN_NUMBER = {RUN_NUMBER + 1} and re-run")
    print(f"{'='*55}\n")


  Retail Data Generator — Run 2
  Total rows per table : 2000
  Expected rejects     : ~596 per table

[1/3] Generating customers...
  ✓ Uploaded → adls://retail-analytics/landing/customers/customers_run2.csv
[2/3] Generating products...
  ✓ Uploaded → adls://retail-analytics/landing/products/products_run2.csv
[3/3] Generating orders...
  ✓ Uploaded → adls://retail-analytics/landing/orders/orders_run2.csv

  All done for Run 2!
  customers → adls://retail-analytics/landing/customers/customers_run2.csv
  products  → adls://retail-analytics/landing/products/products_run2.csv
  orders    → adls://retail-analytics/landing/orders/orders_run2.csv

  Next day → set RUN_NUMBER = 3 and re-run

